# Attention Demo

In [ ]:
!nvidia-smi

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import pickle
import os

sys.path.append("../../dev")
from utils import *
from model import *
from prefixes import *
from metrics import *
from visualize import *

sys.path.append("../../model")
from model_params import *

sys.path.append("../../data")
from dataset_params import *
from prompt_params import *
from demo_params import *

## 1) Setup

In [ ]:
model = "gpt_j"
dataset = "unnatural"
setting = "permuted_incorrect_labels"
prompt_format_indx = 0 # use first prompt format
num_inputs = 250
num_demos = 40

In [ ]:
model_params = MODEL_PARAMS[model]
dataset_params = DATASET_PARAMS[dataset]
prompt_params = PROMPT_PARAMS[dataset][prompt_format_indx]
demo_params = DEMO_PARAMS[setting]

## 2) Get Model and Tokenizer

In [ ]:
file_path = model_params["file_path"]
model, tokenizer = get_model_and_tokenizer(file_path)

## 3) Get Dataset

In [ ]:
print("*Fetch data.*")

dataset = get_dataset(dataset_params)

print(f"There are {len(dataset)} classes.")
print(f"There are {len(dataset[0])} examples per class.")

## 4) Construct Prefixes

In [ ]:
print("*Build prefixes.*")

num_inputs = 250
num_demos = 40
prefixes = Prefixes(dataset, prompt_params, demo_params, model_params, tokenizer, num_inputs, num_demos)

print("True prefix example:")
print("--------------------")
print(prefixes.true_prefixes[0])

print("False prefix example:")
print("---------------------")
print(prefixes.false_prefixes[0])

## 5) Get Attention Weights

In [ ]:
print("*Run inference and get attention weights.*")

attn_weights = get_attn_weights(
    model,
    prefixes,
)

n_pfx, n_inputs, n_layers, n_heads, n_tokens, _ = attn_weights.shape
print(
    f"For n_prfx (e.g. {n_pfx}), n_inputs (e.g. {n_inputs}) we have the "
    f"attention weights for all n_layers (e.g. {n_layers}), n_heads (e.g. "
    f"{n_heads}), and n_tokens x n_tokens ({n_tokens} x {n_tokens})."
)

## 6) Metrics: Context Following Scores

In [ ]:
print("*Compute context following scores.*")
labels = prefixes.true_prefixes_labels + prefixes.false_prefixes_labels
tok_label_indx = (
    prefixes.true_prefixes_tok_label_indx + prefixes.false_prefixes_tok_label_indx
)
label_indx_vals = []
for indices, labs in zip(tok_label_indx, labels):
    inpt = []
    for indxx, lab in zip(indices, labs):
        cntxt = []
        for indx in indxx:
            cntxt.append((indx, lab))
        inpt.append(cntxt)
    label_indx_vals.append(inpt)
n_labels = len(prompt_params["labels"])
context_following_scores = get_context_following_scores(
    attn_weights, label_indx_vals, n_labels
)

In [ ]:
print("*Average metrics over inputs and massage data into df.*")

indx = ["prefix_type", "n_inputs", "demo_indx", "layer_indx", "head_indx"]
metrics = [
    "cfs_lab_same",
    "cfs_lab",
    "cfs_prec_lab_same",
    "cfs_prec_lab",
    "cfs_lab_ratio",
    "cfs_prec_lab_ratio",
    "cfs_lab_prime",
    "cfs_prec_lab_prime",
]
group_col_names = ["demo_indx", "prefix_type", "layer_indx", "head_indx"]
attn_metrics_df = get_attn_metrics_df(
    context_following_scores, indx, metrics, group_col_names, "mean"
).fillna(0)

In [ ]:
attn_metrics_df.head()

## 7) Plot Metrics

In [ ]:
metrics = [
    "cfs_lab_same",
    "cfs_lab",
    "cfs_prec_lab_same",
    "cfs_prec_lab",
    "cfs_lab_ratio",
    "cfs_prec_lab_ratio",
    "cfs_lab_prime",
    "cfs_prec_lab_prime",
]

top_k = 100
filters = {
    "cfs_lab_ratio": ("cfs_lab", top_k),
    "cfs_prec_lab_ratio": ("cfs_prec_lab", top_k),
    "cfs_lab_prime": ("cfs_lab", top_k),
    "cfs_prec_lab_prime": ("cfs_prec_lab", top_k),
}

demo_indx = model_params["max_demos"] - 1

title_params = {"model": "gpt_j", "dataset": "unnatural", "prompt_indx": 0}

In [ ]:
plot_attn_metrics_detailed(
    attn_metrics_df.copy(), metrics, filters, demo_indx, title_params
)

In [ ]:
plot_attn_metrics_compressed(
    attn_metrics_df.copy(), metrics, filters, demo_indx, title_params
)